In [1]:
import requests, json, sys

AURA_URL = "https://azcarecheck.azdhs.gov/s/sfsites/aura"

# ==== PASTE these exactly from DevTools (Form Data) ====
RAW_MESSAGE = r'''
{"actions":[{"id":"71;a","descriptor":"aura://ApexActionController/ACTION$execute","callingDescriptor":"UNKNOWN","params":{"namespace":"","classname":"AZCCFacilityHeaderController","method":"getCustomMetadataInfo","params":{"accountId":"001cs00000Wo4UiAAJ","licenseId":null},"cacheable":false,"isContinuation":false}},{"id":"72;a","descriptor":"aura://ApexActionController/ACTION$execute","callingDescriptor":"UNKNOWN","params":{"namespace":"","classname":"AZCCFacilityDetailsTabController","method":"getFacilityDetails","params":{"facilityId":"001cs00000Wo4UiAAJ"},"cacheable":false,"isContinuation":false}}]}'''
AURA_CONTEXT = r'''{"mode":"PROD","fwuid":"THl4S21tS3lfX1VPdk83d1ZYQXI4UUo4d1c2djVyVVc3NTc1a1lKNHV4S3cxMy4zMzU1NDQzMi4yNTE2NTgyNA","app":"siteforce:communityApp","loaded":{"APPLICATION@markup://siteforce:communityApp":"1407_X7CThrr6Nu7XUXxm8uweXA"},"dn":[],"globals":{},"uad":true}'''
PAGE_URI = "/s/facility-details?facilityId=001cs00000Wo4UiAAJ"  # paste if different
AURA_TOKEN = ""  # paste if present (often empty)

# ==== Optional but recommended: copy these from Request Headers ====
HEADERS = {
    "Content-Type": "application/x-www-form-urlencoded;charset=UTF-8",
    "Origin": "https://azcarecheck.azdhs.gov",
    "Referer": "https://azcarecheck.azdhs.gov" + PAGE_URI,
    "User-Agent": "Mozilla/5.0",
}

# ---- helper: update facilityId inside the message JSON safely ----
def update_facility_id_in_message(raw_message: str, new_facility_id: str) -> str:
    msg = json.loads(raw_message)
    # Typical path: actions[0].params.params.facilityId
    try:
        msg["actions"][0]["params"]["params"]["facilityId"] = new_facility_id
    except Exception as e:
        print("Could not update facilityId in message; check the structure.", file=sys.stderr)
        raise
    return json.dumps(msg, separators=(",", ":"))

def call_aura(message_json: str, context_json: str, page_uri: str, token: str):
    payload = {
        "message": message_json,
        "aura.context": context_json,
        "aura.pageURI": page_uri,
        "aura.token": token,
    }
    resp = requests.post(AURA_URL, data=payload, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    text = resp.text.lstrip()  # handle anti-XSSI like "for(;;);"
    if text.startswith("for(;;);"):
        text = text[len("for(;;);"):]
    data = json.loads(text)

    if "actions" not in data:
        # Print a helpful snippet so you can see what's returned
        print("Unexpected response; first 600 chars:", text[:600], file=sys.stderr)
        raise KeyError("'actions' not in response")
    return data["actions"][0]

# ---- example usage: change the ID and fetch ----
if __name__ == "__main__":
    facility_id = "001cs00000Wo4UiAAJ"  # change as needed
    msg_for_id = update_facility_id_in_message(RAW_MESSAGE, facility_id)
    action = call_aura(msg_for_id, AURA_CONTEXT, PAGE_URI, AURA_TOKEN)

    # Happy path: the payload is usually in returnValue
    if "returnValue" in action:
        result = action["returnValue"]
        print(json.dumps(result, indent=2))
        with open("facility_details.json", "w") as f:
            json.dump(result, f, indent=2)
    else:
        # Show what we got so you can adjust
        print(json.dumps(action, indent=2))



KeyError: 'actions'